# Credit Card Fraud Detection

Predicting fraudulent transactions in a heavily imbalanced dataset
(0.17% fraud). The goal isn't raw accuracy as a do-nothing model 
(only predict legitimate) scores 99.83%.The goal is catching fraud, 
measured by precision and recall.

**My Approach:** naive baseline (v1) → diagnose why accuracy misleads →
improved model with imbalance handling and threshold tuning (v2) →
compare against a random forest → ship the best trade-off.

In [1]:
import pandas as pd
df = pd.read_csv("../data/creditcard.csv")
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [2]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [3]:
df['Class'].value_counts(normalize = True) * 100

Class
0    99.827251
1     0.172749
Name: proportion, dtype: float64

In [4]:
df.groupby('Class')['Amount'].describe()

,count,mean,std,min,25%,50%,75%,max
Class,,,,,,,,
0,284315.0,88.291022,250.105092,0.0,5.65,22.00,77.05,25691.16
1,492.0,122.211321,256.683288,0.0,1.00,9.25,105.89,2125.87


In [5]:
df.groupby('Class').mean()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
Class,,,,,,,,,,,,,,,,,,,,,
0,94838.202258,0.008258,-0.006271,0.012171,-0.007860,0.005453,0.002419,0.009637,-0.000987,0.004467,...,-0.000644,-0.001235,-0.000024,0.000070,0.000182,-0.000072,-0.000089,-0.000295,-0.000131,88.291022
1,80746.806911,-4.771948,3.623778,-7.033281,4.542029,-3.151225,-1.397737,-5.568731,0.570636,-2.581123,...,0.372319,0.713588,0.014049,-0.040308,-0.105130,0.041449,0.051648,0.170575,0.075667,122.211321


In [6]:
df.isnull().sum().sum()

np.int64(0)

## 2. Train/test split

Stratified split so the 0.17% fraud rate is preserved in both sets —
otherwise the test set might get too few frauds to evaluate fairly.

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis = 1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [8]:
y_train.value_counts(normalize=True) * 100


Class
0    99.827075
1     0.172925
Name: proportion, dtype: float64

In [9]:
y_test.value_counts(normalize=True) * 100

Class
0    99.827955
1     0.172045
Name: proportion, dtype: float64

## 3. v1 — naive baseline

Plain logistic regression, no imbalance handling. Expected to fail —
this is the baseline the rest of the project improves on.

In [10]:
from sklearn.linear_model import LogisticRegression

model_v1 = LogisticRegression(max_iter=1000)
model_v1.fit(X_train, y_train)

c:\Users\okata\OneDrive\Documents\Projects\Fraud Detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

## 4. Diagnosis: why 99.9% accuracy is meaningless

v1 scored 99.9% accuracy but caught only 72% of fraud (recall), missing
27 of 98. Accuracy rewards ignoring the rare class, so from here I judge
models on precision (of flagged transactions, how many were fraud) and
recall (of real frauds, how many I caught).

In [11]:
from sklearn.metrics import accuracy_score

y_pred = model_v1.predict(X_test)
accuracy_score(y_test, y_pred)

0.9993328885923949

In [12]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model_v1.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[56853    11]
 [   27    71]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.87      0.72      0.79        98

    accuracy                           1.00     56962
   macro avg       0.93      0.86      0.89     56962
weighted avg       1.00      1.00      1.00     56962



## 5. v2 — handling the imbalance

`class_weight='balanced'` makes missed fraud expensive during training.
This lifts recall to 92% but floods false positives — I then scale the
features (fixing a convergence problem) and tune the decision threshold
to walk the precision/recall trade-off.

In [13]:
model_v2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_test)
print(confusion_matrix(y_test, y_pred_v2))
print(classification_report(y_test, y_pred_v2))

[[55245  1619]
 [    8    90]]
              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.05      0.92      0.10        98

    accuracy                           0.97     56962
   macro avg       0.53      0.94      0.54     56962
weighted avg       1.00      0.97      0.98     56962



c:\Users\okata\OneDrive\Documents\Projects\Fraud Detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
model_v2 = LogisticRegression(max_iter=1000, class_weight='balanced')
model_v2.fit(X_train_scaled, y_train)

y_pred_v2 = model_v2.predict(X_test_scaled)
print(confusion_matrix(y_test, y_pred_v2))
print(classification_report(y_test, y_pred_v2))

[[55478  1386]
 [    8    90]]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [16]:
y_probs = model_v2.predict_proba(X_test_scaled)[:, 1]

threshold = 0.5
y_pred_custom = (y_probs >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[55478  1386]
 [    8    90]]
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.98     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.98      0.99     56962



In [17]:
y_probs = model_v2.predict_proba(X_test_scaled)[:, 1]

threshold = 0.9
y_pred_custom = (y_probs >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[56599   265]
 [   11    87]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.25      0.89      0.39        98

    accuracy                           1.00     56962
   macro avg       0.62      0.94      0.69     56962
weighted avg       1.00      1.00      1.00     56962



In [18]:
y_probs = model_v2.predict_proba(X_test_scaled)[:, 1]

threshold = 0.95
y_pred_custom = (y_probs >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[56728   136]
 [   11    87]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.39      0.89      0.54        98

    accuracy                           1.00     56962
   macro avg       0.69      0.94      0.77     56962
weighted avg       1.00      1.00      1.00     56962



## 6. Comparing models

Comparing logistic regression to a random forest using PR-AUC — a
threshold-independent metric, honest under heavy imbalance (unlike
ROC-AUC). Higher PR-AUC means better trade-offs at every threshold.

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# train the forest (class_weight balanced again, to handle imbalance)
model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
model_rf.fit(X_train_scaled, y_train)

# get fraud probabilities from each model
probs_lr = model_v2.predict_proba(X_test_scaled)[:, 1]
probs_rf = model_rf.predict_proba(X_test_scaled)[:, 1]

# PR-AUC for each — the threshold-independent comparison
print("Logistic Regression PR-AUC:", average_precision_score(y_test, probs_lr))
print("Random Forest PR-AUC:", average_precision_score(y_test, probs_rf))

Logistic Regression PR-AUC: 0.7189705771419241
Random Forest PR-AUC: 0.862939836265611


## 7. Final model and operating point

Random forest wins (PR-AUC 0.86 vs 0.72). Shipping it at threshold 0.3:
85% recall, 83% precision — nearly the same fraud caught as tuned
logistic regression but with ~8× fewer false alarms.

In [20]:
threshold = 0.5
y_pred_rf = (probs_rf >= threshold).astype(int)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[56855     9]
 [   20    78]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.90      0.80      0.84        98

    accuracy                           1.00     56962
   macro avg       0.95      0.90      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [21]:
threshold = 0.3
y_pred_rf = (probs_rf >= threshold).astype(int)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[56847    17]
 [   15    83]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.83      0.85      0.84        98

    accuracy                           1.00     56962
   macro avg       0.91      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962



In [22]:
threshold = 0.7
y_pred_rf = (probs_rf >= threshold).astype(int)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[56862     2]
 [   24    74]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.97      0.76      0.85        98

    accuracy                           1.00     56962
   macro avg       0.99      0.88      0.93     56962
weighted avg       1.00      1.00      1.00     56962



## 8. Save artifacts for deployment

Persist the trained model and the fitted scaler so the endpoint can load
them. Both are needed: incoming transactions must be scaled with THIS
scaler before the model sees them.

In [23]:
import joblib
import os

# make a folder to hold the deployable artifacts
os.makedirs("../models", exist_ok=True)

# save the winning model and the scaler it depends on
joblib.dump(model_rf, "../models/fraud_model.joblib")
joblib.dump(scaler, "../models/scaler.joblib")

print("Saved model and scaler.")

Saved model and scaler.


In [24]:
# sanity check: load them back and confirm they work
loaded_model = joblib.load("../models/fraud_model.joblib")
loaded_scaler = joblib.load("../models/scaler.joblib")

# take one real transaction, scale it, predict on it
sample = X_test.iloc[[0]]                          # one row, raw
sample_scaled = loaded_scaler.transform(sample)     # scale with loaded scaler
prob = loaded_model.predict_proba(sample_scaled)[:, 1][0]
print("Fraud probability:", prob, "→ flagged?" , prob >= 0.3)

Fraud probability: 0.0 → flagged? False


In [26]:
# grab a known fraud from the test set to test the endpoint with
fraud_row = X_test[y_test == 1].iloc[0]
print(fraud_row.to_json())

{"Time":57007.0,"V1":-1.2712441917,"V2":2.4626752685,"V3":-2.8513950033,"V4":2.3244800653,"V5":-1.3722448898,"V6":-0.9481956865,"V7":-3.0652343617,"V8":1.1669269479,"V9":-2.2687705884,"V10":-4.8811429269,"V11":2.2551474887,"V12":-4.6863868976,"V13":0.6523746685,"V14":-6.174288348,"V15":0.594379608,"V16":-4.8496923871,"V17":-6.5365207353,"V18":-3.1190938816,"V19":1.7154944198,"V20":0.5604780757,"V21":0.6529410513,"V22":0.0819309764,"V23":-0.2213478312,"V24":-0.5235821592,"V25":0.2242281619,"V26":0.7563345227,"V27":0.6328004773,"V28":0.2501870928,"Amount":0.01}


In [27]:
legit_row = X_test[y_test == 0].iloc[0]
print(legit_row.to_json())

{"Time":160760.0,"V1":-0.6744660646,"V2":1.4081050197,"V3":-1.1106220536,"V4":-1.3283657784,"V5":1.3889960325,"V6":-1.3084390671,"V7":1.8858789027,"V8":-0.6142329663,"V9":0.3116522125,"V10":0.6507570036,"V11":-0.8577846615,"V12":-0.2299614458,"V13":-0.1998170048,"V14":0.2663713263,"V15":-0.0465441685,"V16":-0.7413980897,"V17":-0.6056166441,"V18":-0.3925681879,"V19":-0.162648311,"V20":0.3943218208,"V21":0.0800842396,"V22":0.8100335956,"V23":-0.2243272304,"V24":0.7078992374,"V25":-0.1358370227,"V26":0.0451021965,"V27":0.5338372191,"V28":0.2913192526,"Amount":23.0}
